# K4b — Reference calibrations: does every sampler find the same posterior?

The user's goal (2026-10-01): recover the original's posterior quickly and reliably, by any
sampler. The posterior is fixed — same priors, targets and likelihood as the original, equal to
estival's at the golden parameters to 1e-6 (notebook 04). This page compares four long
reference runs of the base case, each a different sampler behind the same front end
(`kiribati_tb.reference`), with each other and with the paper's published posterior.

| arm | sampler | chains | where it runs |
| --- | --- | --- | --- |
| `nuts_td8` | dense-mass NUTS, tree depth ≤ 8, ridge-sheared coordinates | 8, one per core | `scripts/cluster/submit_reference.sh nuts_td8` |
| `nuts_td5` | the same, tree depth ≤ 5 | 8, one per core | `... nuts_td5` |
| `sa` | Sample Adaptive MCMC (gradient-free) | 8, one per core | `... sa` |
| `ess` | ensemble slice sampling, differential move (gradient-free) | 64 walkers, vectorised | `... ess` |
| `published` | the paper's DEMetropolisZ, draws 10,000–19,999 | 8 | `reference/published/idata.nc` |

Every arm stops at split R-hat ≤ 1.01 and bulk and tail ESS ≥ 400 for every parameter, or at
its wall-time budget, and can be read at any checkpoint before that.

**Claims to check**

1. Gradients are cheap enough for NUTS: a reverse-mode gradient costs about 4 log-density
   evaluations.
2. The posterior has **one mode** and a **thin, curved ridge** (transmission rate against its
   population scaling), which is why a random-walk sampler mixes slowly.
3. Each arm that reports `diagnostics` has R-hat ≤ 1.01 and ESS ≥ 400 everywhere.
4. Converged arms **agree with each other** within Monte Carlo error, quantile by quantile.
5. The published posterior agrees within **its** (large) Monte Carlo error, and through the port
   it reproduces the paper's headline numbers; the arms' projections are compared with it.
6. Which sampler is cheapest per effective draw (ESS per CPU hour).

The runs are not executed here. Run them on MASSIVE (README, *Running the reference
calibrations on MASSIVE*), copy `outputs/reference/` back, then
`pixi run python scripts/compare_reference.py` and `pixi run python scripts/compare_posteriors.py`.

In [ ]:
import json

import numpyro  # noqa: F401 - import before arviz; see docs/summer4-workarounds.md, S4
import arviz as az
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4.epi.calibration import workflow as wf

from kiribati_tb.calibration import bayesian_model, calibration_setup
from kiribati_tb.paths import REPO_ROOT, compare_dir, reference_root, run_name
from kiribati_tb.pipeline import distinct_optima

OUT = REPO_ROOT / "outputs"
# Which set of reference runs to read: set RUN before starting Jupyter (RUN=r2 pixi run notebook)
# or assign it here, e.g. RUN = "r2". "" reads the unnamed set in outputs/reference/.
RUN = run_name()
REF = reference_root(RUN)
CMP = compare_dir("compare_reference", RUN)
PROJ = compare_dir("compare", RUN)
PUBLISHED = REPO_ROOT / "reference" / "published" / "idata.nc"
# Hook: any further posterior (another published run, a sensitivity analysis) joins every
# comparison below with  scripts/compare_reference.py --arm NAME=path/to/idata.nc
# (and scripts/compare_posteriors.py --arm NAME=... for its projections).

needed = {
    OUT / "bench" / "gradients.json": "pixi run python scripts/bench_gradients.py",
    CMP / "efficiency.csv": f"pixi run python scripts/compare_reference.py --run '{RUN}'",
    PROJ / "projections.parquet": f"pixi run python scripts/compare_posteriors.py --run '{RUN}'",
}
missing = {str(p.relative_to(REPO_ROOT)): c for p, c in needed.items() if not p.exists()}
if missing:
    raise FileNotFoundError("Produce these first:\n" + "\n".join(f"{p}: {c}" for p, c in missing.items()))

setup = calibration_setup()
bm = bayesian_model(setup)
PARAMS = sorted(bm.prior_names())
efficiency = pd.read_csv(CMP / "efficiency.csv")
diag = pd.read_csv(CMP / "diagnostics.csv")
quantiles = pd.read_csv(CMP / "quantiles.csv")
agreement = pd.read_csv(CMP / "agreement.csv")
ARMS = list(efficiency["arm"])
converged = [a for a in efficiency.loc[efficiency["decision"] == "diagnostics", "arm"]]
efficiency

## 1. What a gradient costs

One log-density evaluation and its gradient at the MAP, for several ways of differentiating
through the adaptive Dopri5 solve (`rtol = atol = 1.4e-4`, the original's calibration
tolerance), measured warm (compilation excluded). The error column is the log density against
a `1e-10` solve.

This table measures the original's Dopri5 setting. `CALIBRATION_SOLVER` is now Bosh3 with a PI
controller told about the yearly mixing jumps and 1,024 checkpoints: faster and more accurate
gradients (`docs/gradient-performance.md`).

**What to check:** reverse-mode default is the cheapest gradient, a few evaluations' worth.

In [ ]:
bench = json.loads((OUT / "bench" / "gradients.json").read_text())
bench.pop("_reference_log_density")
table = pd.DataFrame(bench).T
table = table[[c for c in ("mode", "median_s", "compile_s", "steps", "error_vs_1e-10", "error") if c in table]]
table["median_s"] = table["median_s"].astype(float)
eval_s = float(table.loc["eval", "median_s"])
table["x eval"] = (table["median_s"] / eval_s).round(1)
display(table)
px.bar(table[table["median_s"].notna()].reset_index(), x="index", y="median_s", color="mode",
       title="Seconds per call at the MAP (warm, median)", labels={"index": "variant", "median_s": "seconds"}).show()
assert float(table.loc["rev_default", "median_s"]) < 5 * eval_s

## 2. The shape of the posterior

**One mode.** Each arm starts from L-BFGS optima of the 24 best of 256 Latin-hypercube points;
the bars are the optima of the `nuts_td8` arm, best first. Starts that converged reach the same
log density to within a fraction of a nat (spread along flat directions of one optimum); the
rest stalled tens of nats below.

**A thin, curved ridge.** The Laplace covariance at the mode has one direction with standard
deviation about 0.02 (logit units) against 0.3–1.8 for the rest: mostly `raw_transmission_rate`
against `infection_pop_scale`, because the force of infection scales with
`raw × (pop_2020 / N) ** scale`. In the published draws it is a banana; the black line is the
ridge `raw = raw_c K^-scale` fitted at our mode, which the NUTS arms use to straighten it
(`RidgeShear`, unit Jacobian, posterior unchanged).

**What to check:** one group of optima; one Laplace direction far narrower than the rest; the
ridge line through the published draws.

In [ ]:
# Every arm runs the same design and L-BFGS starts; show the first arm that has them.
src = next(
    REF / arm
    for arm in ("nuts_td8", "nuts_td5", "sa", "ess")
    if (REF / arm / "optima.npz.json").exists()
)
optima = wf.Candidates.load(src / "optima.npz")
frame = optima.to_frame().sort_values("log_density", ascending=False).reset_index(drop=True)
px.bar(frame, y="log_density", title="Log density at each optimised start (best first)",
       labels={"index": "start", "log_density": "log density"}).show()
modes = distinct_optima(optima, tol=2.0)
display(pd.DataFrame([{"log density": m.log_density, "starts": len(m.members)} for m in modes]))
# One mode: every start ends within 1 nat of the best (points along flat directions of the same
# optimum) or more than 20 nats below it (a start that stalled far from the posterior).
ld = np.asarray(optima.log_density)
best = ld.max()
assert np.all((ld >= best - 1.0) | (ld < best - 20.0))
assert np.mean(ld >= best - 1.0) >= 0.5

vals, vecs = np.linalg.eigh(np.load(src / "metric.npy"))
spectrum = pd.Series(np.sqrt(vals), index=[f"d{i}" for i in range(len(vals))])
spectrum.plot.bar(title="Laplace covariance at the mode: sd along each eigenvector",
                  labels={"index": "direction (narrowest first)", "value": "sd (logit units)"}).show()
narrow = pd.Series(vecs[:, 0], index=PARAMS)
display(narrow[narrow.abs() > 0.15].round(2).to_frame("narrowest direction"))
assert spectrum.iloc[0] < 0.1 * spectrum.iloc[1:].median()

shear = json.loads((src / "shear.json").read_text())
pub = az.from_netcdf(PUBLISHED).posterior.isel(draw=slice(10000, None, 20))
scale, raw = pub["infection_pop_scale"].values.ravel(), pub["raw_transmission_rate"].values.ravel()
chain = np.repeat(np.arange(pub.sizes["chain"]), pub.sizes["draw"]).astype(str)
fig = px.scatter(x=scale, y=raw, color=chain, opacity=0.4, log_y=True,
                 title="Published draws (every 20th after burn-in): the transmission ridge",
                 labels={"x": "infection_pop_scale", "y": "raw_transmission_rate", "color": "chain"})
s_line = np.linspace(scale.min(), scale.max(), 50)
fig.add_scatter(x=s_line, y=shear["raw_c"] * np.exp(-shear["log_k"] * s_line), mode="lines",
                name="ridge fitted at our mode", line=dict(color="black"))
fig.show()

## 3. Did each arm converge, and at what cost?

From each arm's last checkpoint: draws per chain, the stop decision (`diagnostics` = converged,
`deadline` = budget spent, resubmit to continue), the worst R-hat, the smallest bulk or tail
ESS, divergences and tree depth (NUTS), and ESS per CPU hour of sampling. The published run's
CPU hours are its recorded 72,130 s on 8 cores.

**What to check:** every arm marked `diagnostics` has R-hat ≤ 1.01 and ESS ≥ 400; the
efficiency bars say which sampler to use for the 16-task grid.

In [ ]:
display(efficiency.round(3))
fig = px.bar(diag, x="parameter", y="rhat", color="arm", barmode="group",
             title="Split R-hat per parameter and arm")
fig.add_hline(y=1.01, line_dash="dash")
fig.update_xaxes(tickangle=60)
fig.show()
long = diag.melt(id_vars=["arm", "parameter"], value_vars=["ess_bulk", "ess_tail"])
fig = px.bar(long, x="parameter", y="value", color="arm", facet_row="variable", barmode="group", log_y=True,
             height=700, title="Bulk and tail ESS per parameter and arm")
fig.add_hline(y=400, line_dash="dash")
fig.update_xaxes(tickangle=60)
fig.show()
px.bar(efficiency.dropna(subset=["ess_min_per_cpu_hour"]), x="arm", y="ess_min_per_cpu_hour", log_y=True,
       title="Smallest ESS per CPU hour of sampling", labels={"ess_min_per_cpu_hour": "ESS per CPU hour"}).show()
for arm in converged:
    d = diag[diag["arm"] == arm]
    assert d["rhat"].max() <= 1.01 and d[["ess_bulk", "ess_tail"]].min().min() >= 400, arm

## 4. Do the arms find the same posterior?

Marginals per arm, each parameter scaled to its prior range (0 = lower bound). Then, for every
pair of arms, each quantile's difference divided by the combined Monte Carlo standard error
(`arviz.mcse(method="quantile")`, which accounts for autocorrelation): |z| ≤ 3 is agreement at
the precision the runs have. The published arm's MCSE is large (ESS 35–419), so it agrees
"cheaply"; a large |z| against it is a real disagreement.

**What to check:** converged arms overlap in every panel and agree pairwise (|z| < 4); any
parameter piled against a prior bound (data pushing against the prior) is listed.

In [ ]:
priors = {p.name: (p.lo, p.hi) for p in bm.site_priors()}
rows, rng = [], np.random.default_rng(0)
for arm in ARMS:
    path = PUBLISHED if arm == "published" else REF / arm / "mcmc" / "idata.nc"
    if not path.exists():
        continue
    post = az.from_netcdf(path).posterior
    if arm == "published":
        post = post.isel(draw=slice(10000, None))
    for p in PARAMS:
        x = rng.choice(post[p].values.ravel(), size=min(post[p].size, 8000), replace=False)
        lo, hi = priors[p]
        rows.append(pd.DataFrame({"arm": arm, "parameter": p, "position": (x - lo) / (hi - lo)}))
marg = pd.concat(rows)
fig = px.histogram(marg, x="position", color="arm", facet_col="parameter", facet_col_wrap=4,
                   histnorm="probability density", barmode="overlay", opacity=0.45, nbins=40, height=1500,
                   title="Posterior marginals by arm, scaled to each prior range")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(range=[0, 1])
fig.show()
near = marg.groupby(["arm", "parameter"])["position"].agg(
    lower=lambda x: (x < 0.05).mean(), upper=lambda x: (x > 0.95).mean())
display(near[(near > 0.25).any(axis=1)].round(2).rename(columns=lambda c: f"share within 5% of {c} bound"))

In [ ]:
display(agreement.round(3))
px.bar(agreement, x=agreement["arm_a"] + " vs " + agreement["arm_b"], y="max_abs_z",
       title="Largest |quantile difference| / combined MCSE per pair of arms",
       labels={"x": "pair", "max_abs_z": "max |z|"}).add_hline(y=3, line_dash="dash").show()
for _, row in agreement.iterrows():
    if row["arm_a"] in converged and row["arm_b"] in converged:
        assert row["max_abs_z"] < 4, (row["arm_a"], row["arm_b"], row["worst"])

## 5. The numbers the paper reports

Each posterior's draws through the baseline and five screening scenarios (published: 2,000
post-burn-in draws, as the paper did; reference arms: 1,000). "Averted" is the relative
reduction in cumulative TB episodes (entries into the two asymptomatic states) or TB deaths
over 2026–2035 against no screening. Medians with equal-tailed 95% intervals; the paper's own
numbers are the first column.

**What to check:** the published posterior through the port reproduces the paper (an end-to-end
check of the port's scenarios); the reference arms' numbers, and how far they move from the
published ones.

In [ ]:
PAPER = {
    ("scenario_3", "episodes_averted_perc"): "61 (59–62)",
    ("scenario_3", "deaths_averted_perc"): "57 (56–57)",
    ("scenario_1", "episodes_averted_perc"): "≈34",
    ("scenario_6", "episodes_averted_perc"): "≈34",
    ("scenario_18", "episodes_averted_perc"): "≈10",
    ("baseline", "prevalence_per100k_2026"): "857 (721–999)",
    ("baseline", "incidence_per100k_2026"): "852 (670–1044)",
    ("baseline", "viable_infection_perc_2026"): "18 (14–23)",
    ("scenario_3", "incidence_per100k_2028"): "220",
    ("scenario_3", "mortality_per100k_2028"): "26",
    ("scenario_3", "incidence_per100k_2035"): "291",
    ("baseline", "incidence_per100k_2035"): "894",
    ("baseline", "unreachable_share_of_incidence_2025"): "20 (19–20)",
    ("scenario_3", "unreachable_share_of_incidence_2028"): "55 (51–58)",
}
proj = pd.read_parquet(PROJ / "projections.parquet")
proj.loc[proj["metric"].str.startswith("unreachable_share"), "value"] *= 100.0


def fmt(s: pd.Series) -> str:
    lo, med, hi = np.quantile(s, [0.025, 0.5, 0.975])
    return f"{med:.1f} ({lo:.1f}–{hi:.1f})"


headline = proj.groupby(["scenario", "metric", "arm"])["value"].apply(fmt).unstack("arm")
headline.insert(0, "paper", [PAPER.get(i, "") for i in headline.index])
display(headline)
avert = proj[proj["metric"].isin(["episodes_averted_perc", "deaths_averted_perc"])]
px.box(avert, x="scenario", y="value", color="arm", facet_col="metric", points=False,
       category_orders={"scenario": ["scenario_1", "scenario_6", "scenario_3", "scenario_8", "scenario_18"]},
       title="Burden averted 2026–2035 (%), by scenario and posterior", labels={"value": "% averted"}).show()
pub = proj[(proj["arm"] == "published") & (proj["scenario"] == "scenario_3") & (proj["metric"] == "episodes_averted_perc")]["value"]
lo, med, hi = np.quantile(pub, [0.025, 0.5, 0.975])
assert abs(med - 61) < 1 and abs(lo - 59) < 1 and abs(hi - 62) < 1

## 6. A cheap alternative that does not work: Laplace + PSIS

A Student-t (5 df) at the mode with the Laplace covariance, corrected by Pareto-smoothed
importance sampling. With a thin curved ridge the proposal misses: the log importance ratios
span thousands of nats and the importance-sampling ESS is about 1 of 4,000.

**What to check:** the ratios' spread; Laplace is not a substitute for MCMC here.

In [ ]:
lap_dir = OUT / "alternatives" / "laplace_psis"
if (lap_dir / "summary.json").exists():
    display(pd.Series(json.loads((lap_dir / "summary.json").read_text())))
    ratio = np.load(lap_dir / "log_ratio.npy")
    px.histogram(x=ratio - ratio.max(), nbins=80, log_y=True,
                 title="Laplace + PSIS: log importance ratios relative to the largest",
                 labels={"x": "log p − log q − max"}).show()

## Notes

- **Likelihood: mean or sum over notification years.** The code that produced the published
  posterior (estival 0.6) averages each target's normal log-densities over its years; the
  paper's methods appendix (§9.2) writes a sum. Only notifications has several years (25), so
  the sum weights that target 25 times more. The port keeps the code's mean, which is what the
  published posterior is; `pixi run calibrate --aggregate sum` runs the other.
- **Ensemble R-hat.** For `ess` the walkers are treated as chains; walkers in one ensemble are
  not independent, so its R-hat is optimistic. Judge it by agreement with the other arms.